# Thực nghiệm K Nearest Neighbors KNN

## Mini Capstone Machine Learning

Notebook này phụ trách KNN trong dự án so sánh ba thuật toán. Mô hình dùng dữ liệu sạch và train/test split cố định từ notebook 02. Siêu tham số được chọn bằng 5-fold cross-validation trên tập train theo F1-macro; tập test chỉ dùng để đánh giá cấu hình KNN tốt nhất một lần cuối.

## 1. KNN và các siêu tham số

KNN dự đoán nhãn của một mẫu dựa trên các hàng xóm gần nhất trong không gian đặc trưng. Vì khoảng cách rất nhạy với thang đo, `StandardScaler` phải nằm trong pipeline.

Các siêu tham số thử nghiệm:

- `n_neighbors (k)`: 3, 5, 7, 9, 11, 15, 21.
- `weights`: `uniform` hoặc `distance`.
- `metric`: `euclidean` hoặc `manhattan`.

Tổng cộng có 28 cấu hình. Cấu hình được chọn theo F1-macro trung bình qua cross-validation; Recall của lớp ác tính M được theo dõi để hỗ trợ phần nhận xét.

In [ ]:
from pathlib import Path
import hashlib
import json
import time

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', palette='Set2')
RANDOM_STATE = 42
CV_FOLDS = 5
# Chạy một worker để tránh lỗi multiprocessing trong một số môi trường Anaconda Windows.
GRID_SEARCH_N_JOBS = 1

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

clean_data_path = project_root / 'data' / 'processed' / 'breast_cancer_clean.csv'
splits_dir = project_root / 'data' / 'splits'
train_indices_path = splits_dir / 'train_indices.csv'
test_indices_path = splits_dir / 'test_indices.csv'
metadata_path = splits_dir / 'split_metadata.json'
output_dir = project_root / 'outputs' / 'knn'

required_paths = [clean_data_path, train_indices_path, test_indices_path, metadata_path]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError('Thiếu file cần thiết:\n' + '\n'.join(missing_paths))

print(f'Dữ liệu sạch: {clean_data_path}')
print(f'Split dùng chung: {splits_dir}')
print(f'Kết quả KNN sẽ lưu tại: {output_dir}')

## 2. Kiểm tra tính nhất quán của dữ liệu và split

Notebook kiểm tra SHA-256 của file sạch với metadata do notebook 02 tạo. Nếu dữ liệu đã thay đổi hoặc thứ tự dòng bị đảo, quá trình dừng để tránh dùng sai index train/test.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(8192), b''):
            digest.update(chunk)
    return digest.hexdigest()

metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
current_sha256 = sha256_file(clean_data_path)
if current_sha256 != metadata['source_sha256']:
    raise ValueError(
        'Dữ liệu sạch đã thay đổi sau khi tạo split. Không thể bảo đảm train/test index còn đúng.'
    )

data = pd.read_csv(clean_data_path)
train_indices = pd.read_csv(train_indices_path)['row_index'].astype(int).tolist()
test_indices = pd.read_csv(test_indices_path)['row_index'].astype(int).tolist()

if len(data) != metadata['total_rows']:
    raise ValueError('Số dòng dữ liệu không trùng metadata của split.')
if set(train_indices).intersection(test_indices):
    raise ValueError('Train và test có index trùng nhau.')
if len(train_indices) != metadata['train_rows'] or len(test_indices) != metadata['test_rows']:
    raise ValueError('Kích thước train/test không trùng metadata.')
if min(train_indices + test_indices) < 0 or max(train_indices + test_indices) >= len(data):
    raise ValueError('Có row_index nằm ngoài phạm vi dữ liệu.')

print('Dữ liệu và split hợp lệ.')
print(f'SHA-256: {current_sha256}')

## 3. Tạo train/test theo index cố định

Notebook này không gọi lại hàm chia dữ liệu. Toàn bộ 30 đặc trưng được giữ lại; `diagnosis` được mã hóa `B=0`, `M=1`.

In [ ]:
train_data = data.iloc[train_indices].copy()
test_data = data.iloc[test_indices].copy()

feature_columns = [column for column in data.columns if column != 'diagnosis']
X_train = train_data[feature_columns]
X_test = test_data[feature_columns]
y_train = train_data['diagnosis'].map({'B': 0, 'M': 1})
y_test = test_data['diagnosis'].map({'B': 0, 'M': 1})

print(f'X_train: {X_train.shape}; X_test: {X_test.shape}')
display(pd.DataFrame({
    'train': train_data['diagnosis'].value_counts(),
    'test': test_data['diagnosis'].value_counts(),
}).fillna(0).astype(int))

## 4. Baseline KNN

Baseline được đặt trước khi tuning: `k=5`, uniform weighting và Euclidean distance. Baseline chỉ được đánh giá bằng cross-validation trên train để làm mốc so sánh với KNN sau tuning.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'f1_macro': 'f1_macro',
    'accuracy': 'accuracy',
    'recall_malignant': make_scorer(recall_score, pos_label=1),
}

baseline_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5, weights='uniform', metric='euclidean')),
])

baseline_cv = cross_validate(
    baseline_pipeline, X_train, y_train, cv=cv, scoring=scoring, return_train_score=False
)
baseline_summary = {
    metric: {
        'mean': float(baseline_cv[f'test_{metric}'].mean()),
        'std': float(baseline_cv[f'test_{metric}'].std()),
    }
    for metric in scoring
}
display(pd.DataFrame(baseline_summary).T.rename(columns={'mean': 'trung_binh_cv', 'std': 'do_lech_chuan_cv'}))

## 5. Tuning siêu tham số KNN bằng GridSearchCV

GridSearchCV sử dụng cùng 5-fold CV. `f1_macro` là tiêu chí `refit`, do đó cấu hình đứng đầu là cấu hình cân bằng chất lượng dự đoán trên hai lớp tốt nhất theo CV. Accuracy và Recall M chỉ là chỉ số bổ sung để phân tích, không thay đổi quy tắc chọn mô hình.

In [ ]:
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier()),
])

parameter_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan'],
}

knn_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=parameter_grid,
    scoring=scoring,
    refit='f1_macro',
    cv=cv,
    return_train_score=False,
    n_jobs=GRID_SEARCH_N_JOBS,
    verbose=2,
)

tuning_start = time.perf_counter()
knn_search.fit(X_train, y_train)
tuning_seconds = time.perf_counter() - tuning_start

print(f'Số cấu hình đã thử: {len(knn_search.cv_results_["params"])}')
print(f'Thời gian tuning: {tuning_seconds:.2f} giây')
print(f'Best parameters: {knn_search.best_params_}')
print(f'Best F1-macro CV: {knn_search.best_score_:.4f}')

### 5.1. Bảng Top 10 cấu hình

Bảng được xếp theo rank F1-macro CV. Chênh lệch nhỏ hơn độ lệch chuẩn CV cần được diễn giải thận trọng; không nên chỉ nhìn vào chữ số thập phân cuối cùng.

In [ ]:
cv_results = pd.DataFrame(knn_search.cv_results_)
tuning_summary = pd.DataFrame({
    'rank_f1_macro': cv_results['rank_test_f1_macro'],
    'k': cv_results['param_knn__n_neighbors'].astype(int),
    'weights': cv_results['param_knn__weights'].astype(str),
    'metric': cv_results['param_knn__metric'].astype(str),
    'mean_f1_macro_cv': cv_results['mean_test_f1_macro'],
    'std_f1_macro_cv': cv_results['std_test_f1_macro'],
    'mean_recall_malignant_cv': cv_results['mean_test_recall_malignant'],
    'mean_accuracy_cv': cv_results['mean_test_accuracy'],
}).sort_values(['rank_f1_macro', 'k']).reset_index(drop=True)

top_10_configurations = tuning_summary.head(10)
display(top_10_configurations.style.format({
    'mean_f1_macro_cv': '{:.4f}',
    'std_f1_macro_cv': '{:.4f}',
    'mean_recall_malignant_cv': '{:.4f}',
    'mean_accuracy_cv': '{:.4f}',
}))

### 5.2. Line chart: k và F1-macro CV

Mỗi biểu đồ thể hiện một metric khoảng cách. Mỗi đường là một cách weighting. Error bar là độ lệch chuẩn F1-macro qua 5 folds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for axis, metric_name in zip(axes, ['euclidean', 'manhattan']):
    metric_results = tuning_summary[tuning_summary['metric'] == metric_name]
    for weights_name, group in metric_results.groupby('weights'):
        group = group.sort_values('k')
        axis.errorbar(
            group['k'],
            group['mean_f1_macro_cv'],
            yerr=group['std_f1_macro_cv'],
            marker='o',
            capsize=4,
            label=weights_name,
        )
    axis.set_title(f'{metric_name.title()} distance')
    axis.set_xlabel('Số hàng xóm k')
    axis.set_xticks([3, 5, 7, 9, 11, 15, 21])
    axis.legend(title='weights')
axes[0].set_ylabel('Mean F1-macro CV')
fig.suptitle('Ảnh hưởng của k, weights và metric đến F1-macro')
fig.tight_layout()
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / 'line_chart_k_f1_macro.png', dpi=200, bbox_inches='tight')
plt.show()

### 5.3. Heatmap cấu hình KNN

Mỗi heatmap cố định một metric khoảng cách; hàng là k, cột là weights và giá trị/màu là Mean F1-macro CV. Heatmap cho phép nhìn nhanh tổ hợp nào tốt hơn trong cùng metric. Biểu đồ được lưu thành `heatmap_euclidean.png` và `heatmap_manhattan.png`.

In [ ]:
for metric_name in ['euclidean', 'manhattan']:
    heatmap_data = (
        tuning_summary[tuning_summary['metric'] == metric_name]
        .pivot(index='k', columns='weights', values='mean_f1_macro_cv')
        .sort_index()
    )
    plt.figure(figsize=(6, 5))
    sns.heatmap(heatmap_data, annot=True, fmt='.4f', cmap='YlGnBu', vmin=0, vmax=1)
    plt.title(f'F1-macro CV - {metric_name.title()} distance')
    plt.xlabel('weights')
    plt.ylabel('Số hàng xóm k')
    plt.tight_layout()
    plt.savefig(output_dir / f'heatmap_{metric_name}.png', dpi=200, bbox_inches='tight')
    plt.show()

### 5.4. Scatter plot: F1-macro và Recall lớp ác tính

Biểu đồ này giúp kiểm tra trade-off: một cấu hình có F1-macro cao nhưng Recall M thấp đáng kể cần được thảo luận kỹ, vì bỏ sót mẫu ác tính là lỗi quan trọng trong bối cảnh bài toán.

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=tuning_summary,
    x='mean_f1_macro_cv',
    y='mean_recall_malignant_cv',
    hue='metric',
    style='weights',
    size='k',
    sizes=(60, 260),
)
plt.title('Trade-off giữa F1-macro và Recall lớp ác tính M')
plt.xlabel('Mean F1-macro CV')
plt.ylabel('Mean Recall M CV')
plt.tight_layout()
plt.savefig(output_dir / 'scatter_f1_vs_recall_malignant.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Đánh giá KNN tốt nhất trên tập test độc lập

Cell này chỉ chạy sau khi GridSearchCV đã chốt cấu hình theo F1-macro CV. Không dùng kết quả test để quay lại chọn lại siêu tham số.

In [ ]:
best_knn = knn_search.best_estimator_
prediction_start = time.perf_counter()
y_pred = best_knn.predict(X_test)
y_probability_m = best_knn.predict_proba(X_test)[:, 1]
prediction_seconds = time.perf_counter() - prediction_start

test_metrics = {
    'accuracy': float(accuracy_score(y_test, y_pred)),
    'precision_malignant': float(precision_score(y_test, y_pred, pos_label=1, zero_division=0)),
    'recall_malignant': float(recall_score(y_test, y_pred, pos_label=1, zero_division=0)),
    'f1_malignant': float(f1_score(y_test, y_pred, pos_label=1, zero_division=0)),
    'f1_macro': float(f1_score(y_test, y_pred, average='macro', zero_division=0)),
    'roc_auc': float(roc_auc_score(y_test, y_probability_m)),
    'tuning_seconds': float(tuning_seconds),
    'test_prediction_seconds': float(prediction_seconds),
}

display(pd.DataFrame([test_metrics]).T.rename(columns={0: 'gia_tri'}).style.format('{:.4f}'))
print('Best parameters:', knn_search.best_params_)

### 6.1. Confusion matrix và ROC curve

Confusion matrix cho biết số mẫu lành/ác tính dự đoán đúng và sai. ROC curve sử dụng xác suất lớp M; ROC-AUC càng gần 1 càng thể hiện khả năng xếp hạng hai lớp tốt.

In [ ]:
fig, axis = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['B - Lành tính', 'M - Ác tính'], cmap='Blues', colorbar=False, ax=axis
)
axis.set_title('Confusion matrix - KNN tốt nhất')
fig.tight_layout()
fig.savefig(output_dir / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

false_positive_rate, true_positive_rate, _ = roc_curve(y_test, y_probability_m)
plt.figure(figsize=(6, 5))
plt.plot(false_positive_rate, true_positive_rate, label=f'KNN (AUC = {test_metrics["roc_auc"]:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Ngẫu nhiên')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate / Recall M')
plt.title('ROC curve - KNN tốt nhất')
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / 'roc_curve.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Lưu số liệu cho notebook tổng hợp

Các file trong `outputs/knn/` là dữ liệu đầu vào của notebook 06. Chỉ sau khi đã lưu các file này, nhóm mới dùng số liệu KNN để so sánh với SVM và Decision Tree.

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)

(output_dir / 'best_parameters.json').write_text(
    json.dumps(knn_search.best_params_, ensure_ascii=False, indent=2), encoding='utf-8'
)
(output_dir / 'baseline_cv_metrics.json').write_text(
    json.dumps(baseline_summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
(output_dir / 'test_metrics.json').write_text(
    json.dumps(test_metrics, ensure_ascii=False, indent=2), encoding='utf-8'
)

tuning_summary.to_csv(output_dir / 'cv_results.csv', index=False)
classification_report_frame = pd.DataFrame(
    classification_report(
        y_test, y_pred, target_names=['B_benign', 'M_malignant'], output_dict=True, zero_division=0
    )
).T
classification_report_frame.to_csv(output_dir / 'classification_report.csv')

test_predictions = pd.DataFrame({
    'row_index': test_indices,
    'actual_diagnosis': test_data['diagnosis'].values,
    'predicted_diagnosis': pd.Series(y_pred).map({0: 'B', 1: 'M'}).values,
    'probability_malignant': y_probability_m,
})
test_predictions.to_csv(output_dir / 'test_predictions.csv', index=False)

print('Đã lưu kết quả KNN:')
for path in sorted(output_dir.iterdir()):
    print('-', path.name)

## 8. Kết luận phần KNN

Khi thuyết trình, cần phân biệt rõ ba cấp kết quả: baseline CV, các cấu hình trong GridSearchCV và kết quả test của đúng một KNN tốt nhất. Kết luận về việc KNN có tốt hơn SVM/Decision Tree hay không chỉ được đưa ra trong notebook tổng hợp, sau khi ba mô hình cùng dùng split này.